<a href="https://colab.research.google.com/github/sifat-lab/ML-powered_renewabl_-energy/blob/main/ml/spikesoil_ml_01_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/spikesoil_data/'

Mounted at /content/drive


In [5]:
import pandas as pd
import numpy as np

DATA = '/content/drive/MyDrive/spikesoil_data/'

# ── ১. দুই বছরের DKASC লোড + concat ──
P_COL = '100_DKA_M1_A_Phase_Active_Power'
G_COL = '101_DKA_WeatherStation_Radiation_Global_Tilted'
T_COL = '101_DKA_WeatherStation_Weather_Temperature_Celsius'

# Added on_bad_lines='skip' to handle inconsistent row lengths in the CSV
df24 = pd.read_csv(DATA + 'Alice_Springs_2024.csv', parse_dates=['timestamp'], on_bad_lines='skip')
df25 = pd.read_csv(DATA + 'Alice_Springs_2025.csv', parse_dates=['timestamp'], on_bad_lines='skip')

df = pd.concat([df24, df25]).sort_values('timestamp').reset_index(drop=True)
df = df[['timestamp', P_COL, G_COL, T_COL]].dropna()

# ── ২. 5-min resample ──
df = df.set_index('timestamp').resample('5min').mean().dropna().reset_index()

# ── ৩. শুধু দিনের বেলা ──
df = df[df[G_COL] > 20].reset_index(drop=True)

# ── ৪. Time features ──
h = df['timestamp'].dt.hour + df['timestamp'].dt.minute/60
df['sin_t'] = np.sin(2*np.pi*h/24)
df['cos_t'] = np.cos(2*np.pi*h/24)
doy = df['timestamp'].dt.dayofyear
df['sin_d'] = np.sin(2*np.pi*doy/365)
df['cos_d'] = np.cos(2*np.pi*doy/365)

# ── ৫. Sliding windows ──
FEATS = [P_COL, G_COL, T_COL, 'sin_t', 'cos_t', 'sin_d', 'cos_d']
W = 12
HORIZONS = [3, 6, 12]

X, Y = [], []
ts = df['timestamp'].values
vals = df[FEATS].values.astype(np.float32)
p = df[P_COL].values.astype(np.float32)

for i in range(W, len(df) - max(HORIZONS)):
    if (ts[i] - ts[i-W]) != np.timedelta64(W*5, 'm'):
        continue
    X.append(vals[i-W:i])
    Y.append([p[i+h_] for h_ in HORIZONS])

X = np.array(X); Y = np.array(Y)
print(f"Windows: {X.shape}, Targets: {Y.shape}")

np.savez_compressed('dkasc_windows.npz', X=X, Y=Y)

Windows: (85268, 12, 7), Targets: (85268, 3)


In [5]:
print(df24.columns.tolist())

['timestamp', '205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received', '205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average', '205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power', '239_DKA_Totals_BESS_Reactive_Power', '239_DKA_Totals_BESS_Active_Power', '239_DKA_Totals_BESS_Apparent_Power', '239_DKA_Totals_BESS_State_of_Charge', '242_DKA_Totals_Grid_Reactive_Power', '242_DKA_Totals_Grid_Active_Power', '242_DKA_Totals_Grid_Apparent_Power', '241_DKA_Totals_PV_Reactive_Power', '241_DKA_Totals_PV_Active_Power', '241_DKA_Totals_PV_Apparent_Power', '240_DKA_Totals_Site_Demand_Reactive_Power', '240_DKA_Totals_Site_Demand_Active_Power', '240_DKA_Totals_Site_Demand_Apparent_Power', '100_DKA_M1_A_Phase_Active_Energy_Delivered_Received', '100_DKA_M1_A_Phase_Current_Phase_Average', '100_DKA_M1_A_Phase_Active_Power', '100_DKA_M1_A_Phase_Performance_Ratio', '103_DKA_M1_B_Phase_Active_Energy_Delivered_Received', '103_DKA_M1_B_Phase_Current_Phase_Average', '103_DKA_M1_B_Pha

In [6]:
# Persistence: P(t+h) = P(t) — "কিছুই বদলাবে না" অনুমান
p_now = X[:, -1, 0]                       # window-র শেষ power
for j, name in enumerate(['15min', '30min', '60min']):
    mae = np.abs(p_now - Y[:, j]).mean()
    print(f"Persistence MAE @{name}: {mae:.4f} kW")

Persistence MAE @15min: 0.1112 kW
Persistence MAE @30min: 0.1559 kW
Persistence MAE @60min: 0.2412 kW


In [7]:
import torch
import torch.nn as nn

# ── Train/test split: সময় ধরে (2024→train, 2025→test) ──
# X বানানোর সময় ক্রম রক্ষা হয়েছে, তাই প্রথম ~অর্ধেক=2024
split = int(len(X) * 0.5)          # সীমানাটা মোটামুটি বছর-বদল
X_tr, X_te = X[:split], X[split:]
Y_tr, Y_te = Y[:split], Y[split:]

# ── Normalize: train stats দিয়েই দুটোই ──
mu  = X_tr.reshape(-1, X.shape[-1]).mean(0)
sig = X_tr.reshape(-1, X.shape[-1]).std(0) + 1e-6
Xn_tr = (X_tr - mu) / sig
Xn_te = (X_te - mu) / sig
# target-ও normalize (শেখা সহজ হয়), পরে ফিরিয়ে আনব
y_mu, y_sig = Y_tr.mean(), Y_tr.std()
Yn_tr = (Y_tr - y_mu) / y_sig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)   # Colab-এ Runtime→Change runtime type→GPU নিলে 'cuda'

class ForecastGRU(nn.Module):
    def __init__(self, n_feat=7, hidden=32, n_out=3):
        super().__init__()
        self.gru  = nn.GRU(n_feat, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_out)
    def forward(self, x):
        _, h = self.gru(x)          # h: শেষ hidden state = পুরো ঘণ্টার সারাংশ
        return self.head(h.squeeze(0))

model = ForecastGRU().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")
# ~4-5k params — MCU-friendly সাইজ, ইচ্ছা করেই ছোট রাখছি

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.L1Loss()                 # L1 = MAE — যেটা মাপছি সেটাই optimize

Xt = torch.tensor(Xn_tr).to(device)
Yt = torch.tensor(Yn_tr).to(device)
Xv = torch.tensor(Xn_te).to(device)

BATCH = 512
for epoch in range(15):
    model.train()
    perm = torch.randperm(len(Xt))
    tot = 0
    for i in range(0, len(Xt), BATCH):
        idx = perm[i:i+BATCH]
        opt.zero_grad()
        loss = lossf(model(Xt[idx]), Yt[idx])
        loss.backward()
        opt.step()
        tot += loss.item() * len(idx)
    # ── প্রতি epoch-এ test MAE (আসল kW-তে) ──
    model.eval()
    with torch.no_grad():
        pred = model(Xv).cpu().numpy() * y_sig + y_mu   # denormalize
    mae = np.abs(pred - Y_te).mean(0)
    print(f"ep{epoch+1:02d} train={tot/len(Xt):.4f} | test MAE kW: "
          f"15m={mae[0]:.4f} 30m={mae[1]:.4f} 60m={mae[2]:.4f}")

cuda
4035 parameters
ep01 train=0.4825 | test MAE kW: 15m=0.0902 30m=0.0948 60m=0.1144
ep02 train=0.2159 | test MAE kW: 15m=0.0799 30m=0.0849 60m=0.0990
ep03 train=0.2002 | test MAE kW: 15m=0.0744 30m=0.0794 60m=0.0923
ep04 train=0.1913 | test MAE kW: 15m=0.0678 30m=0.0730 60m=0.0867
ep05 train=0.1856 | test MAE kW: 15m=0.0653 30m=0.0707 60m=0.0832
ep06 train=0.1821 | test MAE kW: 15m=0.0621 30m=0.0681 60m=0.0807
ep07 train=0.1797 | test MAE kW: 15m=0.0611 30m=0.0672 60m=0.0795
ep08 train=0.1777 | test MAE kW: 15m=0.0600 30m=0.0665 60m=0.0780
ep09 train=0.1760 | test MAE kW: 15m=0.0591 30m=0.0654 60m=0.0764
ep10 train=0.1746 | test MAE kW: 15m=0.0593 30m=0.0650 60m=0.0760
ep11 train=0.1734 | test MAE kW: 15m=0.0592 30m=0.0651 60m=0.0758
ep12 train=0.1724 | test MAE kW: 15m=0.0587 30m=0.0645 60m=0.0752
ep13 train=0.1715 | test MAE kW: 15m=0.0577 30m=0.0637 60m=0.0742
ep14 train=0.1705 | test MAE kW: 15m=0.0576 30m=0.0631 60m=0.0736
ep15 train=0.1697 | test MAE kW: 15m=0.0576 30m=0.0630 

In [8]:
!pip install snntorch --quiet
import snntorch as snn
from snntorch import surrogate

class ForecastSNN(nn.Module):
    def __init__(self, n_feat=7, hidden=64, n_out=3, beta=0.9):
        super().__init__()
        grad = surrogate.fast_sigmoid()
        self.fc1  = nn.Linear(n_feat, hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=grad)
        self.fc2  = nn.Linear(hidden, hidden)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=grad)
        self.fc_out = nn.Linear(hidden, n_out)
        # output: non-spiking integrator — regression-এর জন্য
        self.li_out = snn.Leaky(beta=0.95, spike_grad=grad,
                                threshold=1e9)   # কখনো spike করবে না,
                                                  # শুধু জমাবে (integrator)

    def forward(self, x):            # x: (batch, 12, 7)
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.li_out.init_leaky()
        for t in range(x.shape[1]):  # ১২টা timestep = ১২টা SNN step
            s1, mem1 = self.lif1(self.fc1(x[:, t]), mem1)
            s2, mem2 = self.lif2(self.fc2(s1), mem2)
            _,  mem3 = self.li_out(self.fc_out(s2), mem3)
        return mem3                  # শেষ membrane potential = prediction

snn_model = ForecastSNN().to(device)
print(sum(p.numel() for p in snn_model.parameters()), "parameters")

opt = torch.optim.Adam(snn_model.parameters(), lr=1e-3)
# training loop হুবহু GRU-রটাই — model বদলে snn_model, 20-25 epoch দাও
# (SNN একটু ধীরে শেখে, বেশি epoch প্রাপ্য)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 12.4 MB/s eta 0:00:00
4867 parameters


In [9]:
lossf = nn.L1Loss()

BATCH = 512
for epoch in range(25):
    snn_model.train()
    perm = torch.randperm(len(Xt))
    tot = 0
    for i in range(0, len(Xt), BATCH):
        idx = perm[i:i+BATCH]
        opt.zero_grad()
        loss = lossf(snn_model(Xt[idx]), Yt[idx])
        loss.backward()
        opt.step()
        tot += loss.item() * len(idx)

    snn_model.eval()
    with torch.no_grad():
        # eval-এ batch করে চালাই — 40k+ window একসাথে দিলে GPU মেমরি টানাটানি হতে পারে
        preds = []
        for i in range(0, len(Xv), 4096):
            preds.append(snn_model(Xv[i:i+4096]).cpu().numpy())
        pred = np.concatenate(preds) * y_sig + y_mu
    mae = np.abs(pred - Y_te).mean(0)
    print(f"ep{epoch+1:02d} train={tot/len(Xt):.4f} | test MAE kW: "
          f"15m={mae[0]:.4f} 30m={mae[1]:.4f} 60m={mae[2]:.4f}")

ep01 train=0.3572 | test MAE kW: 15m=0.0978 30m=0.1062 60m=0.1106
ep02 train=0.2256 | test MAE kW: 15m=0.0840 30m=0.0881 60m=0.0975
ep03 train=0.2120 | test MAE kW: 15m=0.0779 30m=0.0820 60m=0.0936
ep04 train=0.2044 | test MAE kW: 15m=0.0756 30m=0.0793 60m=0.0914
ep05 train=0.1988 | test MAE kW: 15m=0.0730 30m=0.0767 60m=0.0887
ep06 train=0.1949 | test MAE kW: 15m=0.0730 30m=0.0765 60m=0.0872
ep07 train=0.1923 | test MAE kW: 15m=0.0715 30m=0.0758 60m=0.0862
ep08 train=0.1899 | test MAE kW: 15m=0.0694 30m=0.0748 60m=0.0823
ep09 train=0.1884 | test MAE kW: 15m=0.0688 30m=0.0722 60m=0.0815
ep10 train=0.1872 | test MAE kW: 15m=0.0700 30m=0.0735 60m=0.0816
ep11 train=0.1857 | test MAE kW: 15m=0.0684 30m=0.0715 60m=0.0808
ep12 train=0.1849 | test MAE kW: 15m=0.0674 30m=0.0706 60m=0.0794
ep13 train=0.1838 | test MAE kW: 15m=0.0675 30m=0.0706 60m=0.0790
ep14 train=0.1831 | test MAE kW: 15m=0.0679 30m=0.0710 60m=0.0784
ep15 train=0.1826 | test MAE kW: 15m=0.0679 30m=0.0709 60m=0.0788
ep16 train

In [10]:
# forward hook দিয়ে spike গুনি
spike_counts = []

def count_spikes(module, inp, out):
    s = out[0] if isinstance(out, tuple) else out
    spike_counts.append(s.detach().mean().item())  # fraction firing

h1 = snn_model.lif1.register_forward_hook(count_spikes)
h2 = snn_model.lif2.register_forward_hook(count_spikes)

snn_model.eval()
with torch.no_grad():
    _ = snn_model(Xv[:4096])

h1.remove(); h2.remove()

sparsity = 1 - np.mean(spike_counts)
print(f"Mean firing rate: {np.mean(spike_counts)*100:.1f}%")
print(f"Sparsity: {sparsity*100:.1f}% of neuron-timesteps are silent")

Mean firing rate: 13.0%
Sparsity: 87.0% of neuron-timesteps are silent


In [11]:
torch.save(model.state_dict(), DATA + 'gru_forecast_v1.pt')
torch.save(snn_model.state_dict(), DATA + 'snn_forecast_v1.pt')
np.savez(DATA + 'norm_stats.npz', mu=mu, sig=sig, y_mu=y_mu, y_sig=y_sig)